# Puxando dados da api

In [0]:
import pandas as pd
#Pegando os dados da API do BCB para a cotacao do dolar
url = "https://api.bcb.gov.br/dados/serie/bcdata.sgs.1/dados?formato=csv&dataInicial=01/01/2016&dataFinal=31/12/2025"
df_bcb = pd.read_csv(url, sep=';')

display(df_bcb)

# Salvando como tabela 

In [0]:
#Transformando o dataframe pandas em um dataframe do spark
df_bcb = spark.createDataFrame(data =df_bcb)
df_bcb.display()

# Modificando o tipo das colunas

### - Verificando a tabela

In [0]:
#Salva a tabela no schema gold
df_bcb.write.mode("overwrite").saveAsTable("lh_nautical.gold_lh_nautical.dim_cotacao_dolar")
dim_cotacao_dolar = spark.table("lh_nautical.gold_lh_nautical.dim_cotacao_dolar")
#Conferindo se a tabela foi salva
display(dim_cotacao_dolar)

### - Modificando o tipo das colunas

In [0]:
from pyspark.sql.functions import regexp_replace, to_date, col
from pyspark.sql import functions as F
from pyspark.sql.types import *
# Alterando das as / para -
dim_cotacao_dolar = dim_cotacao_dolar.withColumn('data', regexp_replace('data', '/', '-'))
dim_cotacao_dolar.write.mode("overwrite").saveAsTable("lh_nautical.gold_lh_nautical.dim_cotacao_dolar")
#Conferindo se a tabela foi salva
dim_cotacao_dolar.display()


### - Modificando a coluna datas

In [0]:
#Alterando o tipo da coluna data para date
dim_cotacao_dolar = dim_cotacao_dolar.withColumn("data", F.date_format(F.to_date(col("data"), "dd-MM-yyyy"), "yyyy-MM-dd").cast("date"))
dim_cotacao_dolar.display()
dim_cotacao_dolar.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("lh_nautical.gold_lh_nautical.dim_cotacao_dolar")

### - Modificando os nomes das colunas

In [0]:
%sql
-- Alterando o nome das colunas
ALTER TABLE lh_nautical.gold_lh_nautical.dim_cotacao_dolar SET TBLPROPERTIES (
  'delta.columnMapping.mode' = 'name'
);
ALTER TABLE lh_nautical.gold_lh_nautical.dim_cotacao_dolar RENAME COLUMN data TO date;
ALTER TABLE lh_nautical.gold_lh_nautical.dim_cotacao_dolar SET TBLPROPERTIES (
  'delta.columnMapping.mode' = 'name'
);
ALTER TABLE lh_nautical.gold_lh_nautical.dim_cotacao_dolar RENAME COLUMN valor TO value;



In [0]:
%sql
-- Conferindo a mudança
SELECT * FROM lh_nautical.gold_lh_nautical.dim_cotacao_dolar

### - Modificando a coluna value

In [0]:
from pyspark.sql.functions import regexp_replace, to_date, col
from pyspark.sql import functions as F
from pyspark.sql.types import *
dim_cotacao_dolar = spark.read.table("lh_nautical.gold_lh_nautical.dim_cotacao_dolar")
# Alterando das as , para . da coluna value e convertendo para decimal
dim_cotacao_dolar = dim_cotacao_dolar.withColumn('value', regexp_replace('value', ',', '.').cast(DecimalType(10,2)))
dim_cotacao_dolar.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("lh_nautical.gold_lh_nautical.dim_cotacao_dolar")
#Conferindo se a tabela foi salva
dim_cotacao_dolar.display()